In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
import itertools
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
import xgboost as xgb
!pip install xgboost optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.6 MB/s eta 0:00:00


Step 1: Load the Data

In [2]:
# Load the datasets
interactions = pd.read_csv('https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/interactions_train.csv')
books = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/items.csv", encoding='utf-8')
submission_sample = pd.read_csv("https://raw.githubusercontent.com/ywh1021/MA2_ML/refs/heads/main/sample_submission.csv")

# Display the first rows of each dataset
display(interactions.head())
display(books.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


Check the Number of interactions, users and books

In [3]:
# check number of interaction, users, books
n_users = interactions.u.nunique()
n_items = books.i.nunique()
print('number of users =', n_users, '| number of books =', n_items)

number of users = 7838 | number of books = 15291


In [4]:
# ==========================================
# TF-IDF Content-Based Feature Engineering
# ==========================================
print("\nProcessing Content Features (TF-IDF)...")

# 1. Handle the missing values identified in the EDA
for col in ['Title', 'Author', 'Publisher', 'Subjects']:
    books[col] = books[col].fillna('')

# 2. Combine metadata into a single string per book
books['combined_features'] = books['Title'] + ' ' + books['Author'] + ' ' + books['Publisher'] + ' ' + books['Subjects']

# use french stop words
french_stop_words = stopwords.words('french')

# 3. Vectorize the text data (Limit features to prevent memory overload)
tfidf = TfidfVectorizer(stop_words=french_stop_words, max_features=30000)
tfidf_matrix = tfidf.fit_transform(books['combined_features'])

# 4. Calculate Item-Item cosine similarity based on content
content_sim_global = cosine_similarity(tfidf_matrix)

# Free up memory
del tfidf_matrix
gc.collect()
print("TF-IDF matrix generated successfully.")


Processing Content Features (TF-IDF)...
TF-IDF matrix generated successfully.


In [5]:
# sort the interactions by user and time stamp
interactions = interactions.sort_values(["u", "t"])
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
display(interactions)

,u,i,t,pct_rank
0,0,0,1.680191e+09,0.040000
1,0,1,1.680783e+09,0.080000
2,0,2,1.680801e+09,0.120000
3,0,3,1.683715e+09,0.160000
4,0,3,1.683715e+09,0.200000
...,...,...,...,...
87042,7836,3471,1.728644e+09,0.666667
87043,7836,3471,1.728644e+09,1.000000
87044,7837,2191,1.728735e+09,0.333333
87045,7837,88,1.728735e+09,0.666667


In [6]:
# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1

    return data_matrix


# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon) # epsilon is for avoiding dividing 0
    return pred.T  # Transpose to get users as rows and items as columns


# Define the function to predict interactions based on user similarity
def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred


# Implement the precision_recall_at_k function
def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        top_k_items = np.argpartition(prediction[user], -k)[-k:]
        relevant_items_in_top_k = np.sum(ground_truth[user][top_k_items])
        total_relevant_items = np.sum(ground_truth[user])

        # Update Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Calculate the average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k

In [7]:
# ==========================================
# Memory-Safe Data Preparation for Optuna
# ==========================================
print("\n=== Preparing Single Validation Set for Optuna Tuning ===")

# 1. use one set of Train/Val date to tune the parameters (80% Train, 20% Val)
val_split_ratio = 0.80
train_mask_opt = interactions["pct_rank"] < val_split_ratio
val_mask_opt = ~train_mask_opt

train_data_opt = interactions[train_mask_opt]
val_data_opt = interactions[val_mask_opt]

# 2. Set matrix for tuning
train_matrix_opt = create_data_matrix(train_data_opt, n_users, n_items)
val_matrix_opt = create_data_matrix(val_data_opt, n_users, n_items)

# 3. Generate CF prediction
item_sim_opt = cosine_similarity(train_matrix_opt.T)
item_pred_opt = item_based_predict(train_matrix_opt, item_sim_opt)
del item_sim_opt

user_sim_opt = cosine_similarity(train_matrix_opt)
user_pred_opt = user_based_predict(train_matrix_opt, user_sim_opt)
del user_sim_opt

content_pred_opt = item_based_predict(train_matrix_opt, content_sim_global)

# 4. XGBoost training data (negative sampling 1:4)
pos_u, pos_i = np.where(train_matrix_opt == 1)
neg_u, neg_i = np.where(train_matrix_opt == 0)

sample_indices = np.random.choice(len(neg_u), size=len(pos_u) * 4, replace=False)
neg_u = neg_u[sample_indices]
neg_i = neg_i[sample_indices]

train_u = np.concatenate([pos_u, neg_u])
train_i = np.concatenate([pos_i, neg_i])
y_train_opt = np.concatenate([np.ones(len(pos_u)), np.zeros(len(neg_u))])

X_train_opt = np.column_stack((
    user_pred_opt[train_u, train_i],
    item_pred_opt[train_u, train_i],
    content_pred_opt[train_u, train_i]
))

# 5. XGBoost testing data (flatten the whole matrix)
X_val_all_opt = np.column_stack((
    user_pred_opt.ravel(),
    item_pred_opt.ravel(),
    content_pred_opt.ravel()
))

# Delete these massive matrix, ensure RAM for Optuna
del item_pred_opt, user_pred_opt, content_pred_opt, train_matrix_opt
gc.collect()

print("Data preparation complete. Memory optimized.")

# ==========================================
# Memory-Safe Optuna Optimization
# ==========================================
print("\n=== Starting Optuna Optimization on Validation Set ===")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'tree_method': 'hist',
        'objective': 'binary:logistic'
    }

    # Train XGBoost
    model = xgb.XGBClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_opt, y_train_opt)

    # Predict and return to matrix shape
    preds = model.predict_proba(X_val_all_opt)[:, 1]
    pred_matrix = preds.reshape(n_users, n_items)

    # evaluate this Validation Set
    prec, rec = precision_recall_at_k(pred_matrix, val_matrix_opt, k=10)

    # Recall
    trial.set_user_attr('recall_at_10', rec)

    # clean prediction matrix for every round
    del preds, pred_matrix
    gc.collect()

    return prec

# Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15) # 15 trial

print("-" * 50)
print("=== Best XGBoost Model Evaluation (Validation Set) ===")
print(f"Best Precision@10: {study.best_value:.4f}")
print(f"Best Recall@10:    {study.best_trial.user_attrs['recall_at_10']:.4f}")
print("\nBest XGBoost Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


=== Preparing Single Validation Set for Optuna Tuning ===


[I 2026-05-14 10:38:30,807] A new study created in memory with name: no-name-3bd643c5-dd7f-4616-9573-7a44ef0a9d7d


Data preparation complete. Memory optimized.

=== Starting Optuna Optimization on Validation Set ===


[I 2026-05-14 10:41:51,140] Trial 0 finished with value: 0.05870119928553524 and parameters: {'n_estimators': 85, 'max_depth': 5, 'learning_rate': 0.04119448583387486, 'subsample': 0.7964333215127473}. Best is trial 0 with value: 0.05870119928553524.
[I 2026-05-14 10:45:10,188] Trial 1 finished with value: 0.060168410308755506 and parameters: {'n_estimators': 129, 'max_depth': 3, 'learning_rate': 0.11292850978402912, 'subsample': 0.9123338177144513}. Best is trial 1 with value: 0.060168410308755506.
[I 2026-05-14 10:47:42,866] Trial 2 finished with value: 0.06016841030875551 and parameters: {'n_estimators': 96, 'max_depth': 3, 'learning_rate': 0.1710169015766252, 'subsample': 0.7733067470013928}. Best is trial 2 with value: 0.06016841030875551.
[I 2026-05-14 10:51:48,287] Trial 3 finished with value: 0.05868844092881151 and parameters: {'n_estimators': 97, 'max_depth': 7, 'learning_rate': 0.02042031691180715, 'subsample': 0.6948525239363738}. Best is trial 2 with value: 0.0601684103087

--------------------------------------------------
=== Best XGBoost Model Evaluation (Validation Set) ===
Best Precision@10: 0.0603
Best Recall@10:    0.3038

Best XGBoost Hyperparameters:
  n_estimators: 71
  max_depth: 3
  learning_rate: 0.18539119574008642
  subsample: 0.797576028599659


In [7]:
print("\n=== Running 5-Fold CV with Best XGBoost Hyperparameters ===")

# 1. Define the best parameters found by Optuna
best_xgb_params = {
    'n_estimators': 71,
    'max_depth': 3,
    'learning_rate': 0.18539119574008642,
    'subsample': 0.797576028599659,
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'random_state': 42,
    'n_jobs': -1
}

k_folds = 5
cv_precisions = []
cv_recalls = []

for fold in range(k_folds):
    print(f"\n--- Processing Fold {fold + 1}/{k_folds} ---")

    # 2. Time-based split for the current fold
    test_lower = fold / k_folds
    test_upper = (fold + 1) / k_folds

    if fold == k_folds - 1:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] <= test_upper)
    else:
        test_mask = (interactions["pct_rank"] >= test_lower) & (interactions["pct_rank"] < test_upper)

    train_mask = ~test_mask

    train_data_cv = interactions[train_mask]
    test_data_cv = interactions[test_mask]

    # Build Data Matrices
    train_matrix_cv = create_data_matrix(train_data_cv, n_users, n_items)
    test_matrix_cv = create_data_matrix(test_data_cv, n_users, n_items)

    # -------------------------------------------
    # Stage 1: Generate CF prediction scores (XGBoost features)
    # -------------------------------------------
    print("Generating Collaborative Filtering & Content features...")

    item_sim_cv = cosine_similarity(train_matrix_cv.T)
    item_pred_cv = item_based_predict(train_matrix_cv, item_sim_cv)
    del item_sim_cv # Free memory immediately

    user_sim_cv = cosine_similarity(train_matrix_cv)
    user_pred_cv = user_based_predict(train_matrix_cv, user_sim_cv)
    del user_sim_cv # Free memory immediately

    # Use the globally generated content similarity matrix
    content_pred_cv = item_based_predict(train_matrix_cv, content_sim_global)

    # -------------------------------------------
    # Stage 2: Negative sampling and Tabular dataset creation
    # -------------------------------------------
    print("Preparing tabular data for XGBoost...")
    pos_u, pos_i = np.where(train_matrix_cv == 1)
    neg_u, neg_i = np.where(train_matrix_cv == 0)

    # 1:4 Negative Sampling ratio to handle data imbalance
    sample_indices = np.random.choice(len(neg_u), size=len(pos_u) * 4, replace=False)
    neg_u = neg_u[sample_indices]
    neg_i = neg_i[sample_indices]

    train_u = np.concatenate([pos_u, neg_u])
    train_i = np.concatenate([pos_i, neg_i])
    y_train = np.concatenate([np.ones(len(pos_u)), np.zeros(len(neg_u))])

    # Stack the CF scores to form the training features (X_train)
    X_train = np.column_stack((
        user_pred_cv[train_u, train_i],
        item_pred_cv[train_u, train_i],
        content_pred_cv[train_u, train_i]
    ))

    # Flatten the entire user-item space for top-K predictions (X_test)
    X_test_all = np.column_stack((
        user_pred_cv.ravel(),
        item_pred_cv.ravel(),
        content_pred_cv.ravel()
    ))

    # -------------------------------------------
    # Stage 3: Train and Predict
    # -------------------------------------------
    print("Training XGBoost model...")
    model = xgb.XGBClassifier(**best_xgb_params)
    model.fit(X_train, y_train)

    print("Predicting and evaluating...")
    # Predict probabilities for the positive class (Label 1)
    preds = model.predict_proba(X_test_all)[:, 1]

    # Reshape the 1D array back into the 2D user-item matrix format
    pred_matrix = preds.reshape(n_users, n_items)

    # Calculate metrics
    prec, rec = precision_recall_at_k(pred_matrix, test_matrix_cv, k=10)
    cv_precisions.append(prec)
    cv_recalls.append(rec)

    print(f"-> Fold {fold + 1} completed | Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # -------------------------------------------
    # Stage 4: Strict Memory Cleanup
    # -------------------------------------------
    # Delete all large arrays specific to this fold to prevent memory leaks
    del item_pred_cv, user_pred_cv, content_pred_cv, train_matrix_cv
    del X_train, y_train, X_test_all, model, preds, pred_matrix
    gc.collect()

# ==========================================
# Final Evaluation Results
# ==========================================
print("\n" + "="*50)
print("=== Final 5-Fold Cross Validation Results ===")
print(f"Average Precision@10: {np.mean(cv_precisions):.4f} ± {np.std(cv_precisions):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recalls):.4f} ± {np.std(cv_recalls):.4f}")
print("="*50)


=== Running 5-Fold CV with Best XGBoost Hyperparameters ===

--- Processing Fold 1/5 ---
Generating Collaborative Filtering & Content features...
Preparing tabular data for XGBoost...
Training XGBoost model...
Predicting and evaluating...
-> Fold 1 completed | Precision@10: 0.0326, Recall@10: 0.1527

--- Processing Fold 2/5 ---
Generating Collaborative Filtering & Content features...
Preparing tabular data for XGBoost...
Training XGBoost model...
Predicting and evaluating...
-> Fold 2 completed | Precision@10: 0.0555, Recall@10: 0.3244

--- Processing Fold 3/5 ---
Generating Collaborative Filtering & Content features...
Preparing tabular data for XGBoost...
Training XGBoost model...
Predicting and evaluating...
-> Fold 3 completed | Precision@10: 0.0490, Recall@10: 0.2383

--- Processing Fold 4/5 ---
Generating Collaborative Filtering & Content features...
Preparing tabular data for XGBoost...
Training XGBoost model...
Predicting and evaluating...
-> Fold 4 completed | Precision@10: 0

In [ ]:
# filter top 10 prediction
def get_top_10_df(prediction, k=10):
    num_users = prediction.shape[0]
    recommendation_list = []

    for user in range(num_users):
        # Step 1: get top 10 index
        top_k_items = np.argsort(prediction[user])[-k:][::-1]

        # Step 2: use " " to separate strings
        rec_str = " ".join(top_k_items.astype(str))

        # Step 3: save user_id and recommendation strings
        recommendation_list.append({
            "user_id": user,
            "recommendation": rec_str
        })

    # Step 4: transform to dataframe
    df = pd.DataFrame(recommendation_list)
    return df